In [1]:
%%writefile /kaggle/working/03c_preprocess_mri_v3.py
import os
import numpy as np
import pandas as pd
import pydicom
import cv2

BASE    = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
SPLIT   = "test"
OUT_DIR = "/tmp/vol_test_v3"
IMG     = 224
DEPTH   = 32
VIEWS   = [("Sagittal", "fs"), ("Coronal", "fs"), ("Axial", "fs"),
           ("Sagittal", "nonfs"), ("Coronal", "nonfs")]


def choose_series(g, series_root):
    g = g.copy()
    g["n"] = [len(os.listdir(os.path.join(series_root, s)))
              if os.path.isdir(os.path.join(series_root, s)) else 0
              for s in g["SeriesInstanceUID"]]
    g = g[g["n"] > 0]
    chosen, used = [], set()
    for plane, kind in VIEWS:
        c = g[(g["Anatomical_Plane"] == plane) & ~g["SeriesInstanceUID"].isin(used)].copy()
        if kind == "fs":
            c["score"] = c["Fluid_Sensitive"] * 2 + c["Fat_Suppression"]
        else:
            c = c[c["Fat_Suppression"] == 0].copy()
            c["score"] = 1 - c["Fluid_Sensitive"]
        c = c.sort_values(["score", "n"], ascending=False)
        s = c["SeriesInstanceUID"].iloc[0] if len(c) else None
        chosen.append(s)
        if s is not None:
            used.add(s)
    return chosen


def load_series(path):
    slices = []
    for f in os.listdir(path):
        if not f.endswith(".dcm"):
            continue
        ds = pydicom.dcmread(os.path.join(path, f))
        try:
            pos = np.array(ds.ImagePositionPatient, dtype=float)
            ori = np.array(ds.ImageOrientationPatient, dtype=float)
            key = float(pos @ np.cross(ori[:3], ori[3:]))
        except Exception:
            key = float(getattr(ds, "InstanceNumber", 0))
        img = ds.pixel_array.astype(np.float32)
        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            img = img.max() - img
        slices.append((key, img))
    slices.sort(key=lambda x: x[0])
    return [s[1] for s in slices]


def square_resize(img, size):
    h, w = img.shape
    m = max(h, w)
    canvas = np.zeros((m, m), np.float32)
    y, x = (m - h) // 2, (m - w) // 2
    canvas[y:y + h, x:x + w] = img
    return cv2.resize(canvas, (size, size), interpolation=cv2.INTER_AREA)


def to_volume(slices):
    vol = np.stack([square_resize(s, IMG) for s in slices])
    fg = vol[vol > 0]
    lo, hi = np.percentile(fg if fg.size else vol, [0.5, 99.5])
    vol = np.clip((vol - lo) / (hi - lo + 1e-6), 0, 1)
    idx = np.linspace(0, len(vol) - 1, DEPTH).round().astype(int)
    return (vol[idx] * 255).astype(np.uint8)


def process_study(study_uid, g):
    series_root = os.path.join(BASE, f"{SPLIT}_series", study_uid)
    chosen = choose_series(g, series_root)
    vol = np.zeros((len(VIEWS), DEPTH, IMG, IMG), np.uint8)
    mask = np.zeros(len(VIEWS), np.uint8)
    for i, s_uid in enumerate(chosen):
        if s_uid is None:
            continue
        try:
            slices = load_series(os.path.join(series_root, s_uid))
            if slices:
                vol[i] = to_volume(slices)
                mask[i] = 1
        except Exception:
            pass
    np.savez_compressed(os.path.join(OUT_DIR, f"{study_uid}.npz"), vol=vol, mask=mask)


Writing /kaggle/working/03c_preprocess_mri_v3.py


In [2]:
import os, gc, glob, importlib.util, traceback
import numpy as np, pandas as pd, torch, pydicom
from tqdm.auto import tqdm

VARIANT = "K8"

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
VOL = {"v1": "/tmp/vol_test", "crop": "/tmp/vol_test_crop", "280": "/tmp/vol_test_280",
       "crop280": "/tmp/vol_test_crop280", "v2": "/tmp/vol_test_v2", "v3": "/tmp/vol_test_v3"}

FILES = {}
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d != "competitions"]
    for f in files:
        if not f.endswith(".npz"):
            FILES.setdefault(f, os.path.join(root, f))

def find_file(name):
    if name not in FILES:
        raise FileNotFoundError(name)
    return FILES[name]

def load_module(name, path):
    print(f"{name:7s} -> {path}")
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    return m

# nama: (backbone, file bobot, versi data, normalisasi kiri-kanan?, ukuran masukan)
DINO = "vit_small_patch14_dinov2.lvd142m"
MODELS = {
    "effb0":                              ("efficientnet_b0", "efficientnet_b0_best.pt",            "v1",      False, 224),
    "dinov2s_student_v1d":                (DINO, "dinov2s_student_v1d_best.pt",                     "v1",      False, 224),
    "dinov2s_student_v1d_lat":            (DINO, "dinov2s_student_v1d_lat_best.pt",                 "v1",      True,  224),
    "dinov2s_student_v1d_lat_s7":         (DINO, "dinov2s_student_v1d_lat_s7_best.pt",              "v1",      True,  224),
    "dinov2s_student_v1d_lat_crop":       (DINO, "dinov2s_student_v1d_lat_crop_best.pt",            "crop",    True,  224),
    "dinov2s_student_v1d_lat_280":        (DINO, "dinov2s_student_v1d_lat_280_best.pt",             "280",     True,  280),
    "dinov2s_student_v1d_lat_crop280":    (DINO, "dinov2s_student_v1d_lat_crop280_best.pt",         "crop280", True,  280),
    "dinov2s_student_v1d_lat_crop280_s7": (DINO, "dinov2s_student_v1d_lat_crop280_s7_best.pt",      "crop280", True,  280),
    "dinov2s_student_v2":                 (DINO, "dinov2s_student_v2_best.pt",                      "v2",      False, 224),
    "effb0_student_v3":                   ("efficientnet_b0", "effb0_student_v3_best.pt",           "v2",      False, 224),
    "dinov2s_student_d32_lat":            (DINO, "dinov2s_student_d32_lat_best.pt",                 "v3",      True,  224),
    "effb0_student_d32_s7":               ("efficientnet_b0", "effb0_student_d32_s7_best.pt",       "v3",      False, 224),
}
SETS = {
    "H1": list(MODELS),                                                       # val 0.8882 | gold 0.8749
    "K8": ["dinov2s_student_v1d_lat", "dinov2s_student_v1d_lat_280", "dinov2s_student_v1d_lat_crop",
           "dinov2s_student_v1d_lat_crop280", "dinov2s_student_v1d_lat_crop280_s7",
           "dinov2s_student_v1d_lat_s7", "effb0", "effb0_student_d32_s7"],     # val 0.8883 | gold 0.8781
}
USE = SETS[VARIANT]

need = sorted({MODELS[n][2] for n in USE})
PP_SRC = {"v1": ("pp1", "03_preprocess_mri.py"), "crop": ("ppc", "03_preprocess_crop.py"),
          "280": ("pp280", "03_preprocess_280.py"), "crop280": ("ppc280", "03_preprocess_crop280.py"),
          "v2": ("pp2", "03b_preprocess_mri_v2.py"), "v3": ("pp3", None)}
PP = {}
for v in need:
    name, fname = PP_SRC[v]
    PP[v] = load_module(name, find_file(fname) if fname else "/kaggle/working/03c_preprocess_mri_v3.py")
tb = load_module("tb", find_file("11_train_student_lat.py"))
LABELS = tb.LABELS
for v in need:
    print(f"IMG {v}: {PP[v].IMG}")

print(f"\nVARIANT {VARIANT}: {len(USE)} model")
for name in USE:
    b, w, v, lat, img = MODELS[name]
    print(f"{name:36s} data={v:7s} lat={lat!s:5s} img={img} -> {find_file(w)}")
print("Data yang perlu diproses:", need, "| GPU:", torch.cuda.device_count())


pp280   -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-28-preprocess-280/03_preprocess_280.py
ppc     -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-27-preprocess-crop/03_preprocess_crop.py
ppc280  -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-31-preprocess-crop280/03_preprocess_crop280.py
pp1     -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-02-preprocess-mri/03_preprocess_mri.py
pp3     -> /kaggle/working/03c_preprocess_mri_v3.py
tb      -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-29-dinov2-crop-280/11_train_student_lat.py
IMG 280: 280
IMG crop: 224
IMG crop280: 280
IMG v1: 224
IMG v3: 224

VARIANT K8: 8 model
dinov2s_student_v1d_lat              data=v1      lat=True  img=224 -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-23-dinov2-lat/dinov2s_student_v1d_lat_best.pt
dinov2s_student_v1d_lat_280          data=280     lat=True  img=280 -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-29-dinov2-crop-280/dinov2s_student_v1d_lat_280_best.pt
dinov2s_student_

In [3]:
import time, multiprocessing as mp
import cv2
cv2.setNumThreads(0)                       # cegah thread OpenCV berebut CPU di setiap proses

series = pd.read_csv(f"{COMP}/test_series.csv")
test = pd.read_csv(f"{COMP}/test.csv")
uids = test["StudyInstanceUID"].tolist()
groups = list(series.groupby("StudyInstanceUID"))
N_PROC = os.cpu_count() or 4
print(f"Studi uji: {len(uids)} | proses paralel: {N_PROC}")

def _run(args):
    ver, uid, g = args
    try:
        PP[ver].process_study(uid, g)
        return None
    except Exception as e:
        return f"{ver} {uid} {type(e).__name__}: {e}"

avail = {}
for ver in need:
    t0 = time.time()
    pp = PP[ver]
    pp.SPLIT, pp.OUT_DIR = "test", VOL[ver]
    os.makedirs(VOL[ver], exist_ok=True)
    jobs = [(ver, uid, g) for uid, g in groups]
    failed = 0
    with mp.get_context("fork").Pool(N_PROC) as pool:
        for r in tqdm(pool.imap_unordered(_run, jobs, chunksize=4), total=len(jobs), desc=f"Preprocess {ver}"):
            if r is not None:
                failed += 1
                print("Gagal:", r)
    avail[ver] = [u for u in uids if os.path.exists(f"{VOL[ver]}/{u}.npz")]
    print(f"{ver}: volume tersedia {len(avail[ver])}/{len(uids)} | gagal {failed} | {(time.time() - t0) / 60:.1f} menit")


Studi uji: 3 | proses paralel: 4


Preprocess 280:   0%|          | 0/3 [00:00<?, ?it/s]

280: volume tersedia 3/3 | gagal 0 | 0.2 menit


Preprocess crop:   0%|          | 0/3 [00:00<?, ?it/s]

crop: volume tersedia 3/3 | gagal 0 | 0.1 menit


Preprocess crop280:   0%|          | 0/3 [00:00<?, ?it/s]

crop280: volume tersedia 3/3 | gagal 0 | 0.1 menit


Preprocess v1:   0%|          | 0/3 [00:00<?, ?it/s]

v1: volume tersedia 3/3 | gagal 0 | 0.1 menit


Preprocess v3:   0%|          | 0/3 [00:00<?, ?it/s]

v3: volume tersedia 3/3 | gagal 0 | 0.1 menit


In [4]:
def center_x(path):
    try:
        f = sorted(glob.glob(path + "/*.dcm"))
        if not f:
            return np.nan, ""
        d = pydicom.dcmread(f[0], stop_before_pixels=True)
    except Exception:
        return np.nan, ""
    tag = str(getattr(d, "Laterality", "") or "").strip().upper()
    try:
        ipp = np.array(d.ImagePositionPatient, float); iop = np.array(d.ImageOrientationPatient, float)
        ps = np.array(d.PixelSpacing, float)
        c = ipp + iop[:3] * ps[1] * d.Columns / 2 + iop[3:] * ps[0] * d.Rows / 2
        return float(c[0]), tag
    except Exception:
        return np.nan, tag

SIDE = {}
for uid, g in tqdm(series.groupby("StudyInstanceUID"), desc="Sisi lutut"):
    xs, tags = [], []
    for s in g[g["Anatomical_Plane"].isin(["Coronal", "Axial"])]["SeriesInstanceUID"]:
        x, t = center_x(f"{COMP}/test_series/{uid}/{s}")
        if not np.isnan(x): xs.append(x)
        if t in ("L", "R"): tags.append(t)
    cx = float(np.median(xs)) if xs else np.nan
    if not np.isnan(cx) and abs(cx) > 20:
        SIDE[uid] = "L" if cx > 0 else "R"
    elif tags:
        SIDE[uid] = max(set(tags), key=tags.count)
    elif not np.isnan(cx):
        SIDE[uid] = "L" if cx > 0 else "R"
    else:
        SIDE[uid] = "R"
FLIP = {u: SIDE.get(u, "R") == "L" for u in uids}
print("Sisi lutut data uji:", pd.Series(SIDE).value_counts().to_dict())


Sisi lutut:   0%|          | 0/3 [00:00<?, ?it/s]

Sisi lutut data uji: {'R': 2, 'L': 1}


In [5]:
member_preds = {}
for name in USE:
    backbone, wfile, ver, lat, img = MODELS[name]
    ok = avail.get(ver, [])
    if not ok:
        print(f"LEWATI {name}: tidak ada volume data {ver}")
        continue
    try:
        d0 = np.load(f"{VOL[ver]}/{ok[0]}.npz")
        n_views, depth, size = int(d0["mask"].shape[0]), int(d0["vol"].shape[1]), int(d0["vol"].shape[-1])
        flip_idx = {3: [1, 2], 5: [1, 2, 4]}[n_views] if lat else []
        ds = tb.KneeDS(ok, np.full((len(ok), len(LABELS)), np.nan, np.float32), VOL[ver], False,
                       FLIP if lat else None, flip_idx)
        bs = 1 if depth > 16 else 2
        dl = torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=False, num_workers=2)

        net = tb.KneeNet(backbone, n_views=n_views, pretrained=False, img_size=img)
        net.load_state_dict(torch.load(find_file(wfile), map_location="cpu"))
        net = net.cuda().eval()
        if "vit" not in backbone:
            net = net.to(memory_format=torch.channels_last)
        p = tb.predict(net, dl)
        del net; gc.collect(); torch.cuda.empty_cache()

        p = pd.DataFrame(p, columns=LABELS); p.insert(0, "StudyInstanceUID", ok)
        member_preds[name] = test[["StudyInstanceUID"]].merge(p, on="StudyInstanceUID", how="left")[LABELS].values
        print(f"{name:36s} (data {ver}: {n_views} view x {depth} irisan x {size} px, masukan {img}, cermin {flip_idx}) | "
              f"rata-rata prediksi {np.nanmean(member_preds[name]):.3f}")
    except Exception as e:
        print(f"LEWATI {name}: {type(e).__name__}: {e}")
        traceback.print_exc()
        gc.collect(); torch.cuda.empty_cache()

print(f"\nModel berhasil: {len(member_preds)}/{len(USE)} -> {list(member_preds)}")
assert len(member_preds) > 0, "Semua model gagal"


dinov2s_student_v1d_lat              (data v1: 3 view x 16 irisan x 224 px, masukan 224, cermin [1, 2]) | rata-rata prediksi 0.257
dinov2s_student_v1d_lat_280          (data 280: 3 view x 16 irisan x 280 px, masukan 280, cermin [1, 2]) | rata-rata prediksi 0.258
dinov2s_student_v1d_lat_crop         (data crop: 3 view x 16 irisan x 224 px, masukan 224, cermin [1, 2]) | rata-rata prediksi 0.271
dinov2s_student_v1d_lat_crop280      (data crop280: 3 view x 16 irisan x 280 px, masukan 280, cermin [1, 2]) | rata-rata prediksi 0.256
dinov2s_student_v1d_lat_crop280_s7   (data crop280: 3 view x 16 irisan x 280 px, masukan 280, cermin [1, 2]) | rata-rata prediksi 0.250
dinov2s_student_v1d_lat_s7           (data v1: 3 view x 16 irisan x 224 px, masukan 224, cermin [1, 2]) | rata-rata prediksi 0.267
effb0                                (data v1: 3 view x 16 irisan x 224 px, masukan 224, cermin []) | rata-rata prediksi 0.182
effb0_student_d32_s7                 (data v3: 5 view x 32 irisan x 224 px

In [6]:
p_ens = np.nanmean(np.stack([member_preds[n] for n in member_preds]), axis=0)
print("VARIANT", VARIANT, "| anggota ensemble:", len(member_preds), "model | bentuk:", p_ens.shape)

sub = pd.DataFrame(p_ens, columns=LABELS)
sub.insert(0, "StudyInstanceUID", uids)
n_missing = sub[LABELS].isna().any(axis=1).sum()
sub[LABELS] = sub[LABELS].fillna(0.5)

sample = pd.read_csv(f"{COMP}/sample_submission.csv")
sub = sub[sample.columns]
sub.to_csv("/kaggle/working/submission.csv", index=False)

print("Studi tanpa prediksi (diisi 0.5):", n_missing)
print("Shape submission:", sub.shape, "| shape sample:", sample.shape)
print(sub.head())


VARIANT K8 | anggota ensemble: 8 model | bentuk: (3, 12)
Studi tanpa prediksi (diisi 0.5): 0
Shape submission: (3, 13) | shape sample: (3, 13)
                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.013351  0.050862   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.439567  0.146783   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.008769  0.048926   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.416096          0.050495   0.270984    0.083508  0.215603   
1         0.393495          0.281586   0.335419    0.225904  0.376222   
2         0.382942          0.195896   0.332332    0.181090  0.647012   

   Effusion  Synovitis   Baker's  Contusion  Fracture  
0  0.501127   0.093564  0.702405   0.090132  0.003530  
1  0.391840   0.593758  0.099893   0.542588  0.173160  
2  0.166408   0.071736  0.077868   0.079261  0.023439  
